<a href="https://colab.research.google.com/github/CoolingVerseOracle/Coolingverse-data/blob/main/03_Air_Quality_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 파일 로드

## 1. 드라이브에서 파일 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 대기 데이터 각 지역 1년치 통합본 제작
-  파일이름만 계속 바꿔서 한 코드로만 진행함.
[에어코리아](https://www.airkorea.or.kr/web/realSearch?pMENU_NO=97) 실시간 자료조회 통해서 각 월별 데이터 수집 후 통합.

In [ ]:
import os
import glob
import pandas as pd

# ==========================================
# 🚀 1. 경로 설정 (현재 화면에 바로 올렸을 때)
# ==========================================
# 코랩 기본 화면에 올린 경우, 그냥 현재 폴더('.')에서 파일을 찾으면 됩니다!
folder_path = './송내대로*.xls'
output_path = './송내대로(도시)_2025_1년통합본.csv'


# ==========================================
# 2. 에어코리아 엑셀 맞춤형 정제 함수
# ==========================================
def clean_air_data(filepath):
    df = pd.read_excel(filepath)
    data_df = df.iloc[5:].copy()

    cleaned = pd.DataFrame()
    cleaned['측정일시'] = data_df.iloc[:, 0].astype(str)
    cleaned['PM10'] = pd.to_numeric(data_df.iloc[:, 2], errors='coerce')
    cleaned['PM25'] = pd.to_numeric(data_df.iloc[:, 4], errors='coerce')
    cleaned['O3'] = pd.to_numeric(data_df.iloc[:, 6], errors='coerce')
    cleaned['NO2'] = pd.to_numeric(data_df.iloc[:, 8], errors='coerce')
    cleaned['CO'] = pd.to_numeric(data_df.iloc[:, 10], errors='coerce')
    cleaned['SO2'] = pd.to_numeric(data_df.iloc[:, 12], errors='coerce')

    cleaned = cleaned[cleaned['측정일시'].str.contains(':', na=False)]
    return cleaned


# ==========================================
# 3. 파일 병합 및 정렬 시작
# ==========================================
file_list = glob.glob(folder_path)
print(f"🔍 폴더 내에서 {len(file_list)}개의 대기질 파일을 발견했습니다!")

dfs_list = []
for file in sorted(file_list):
    try:
        cleaned_df = clean_air_data(file)
        dfs_list.append(cleaned_df)
        print(f"   ㄴ [성공] {os.path.basename(file)} -> {len(cleaned_df)}행 정제 완료")
    except Exception as e:
        print(f"   ❌ [에러] {os.path.basename(file)} 처리 중 오류 발생: {e}")

if dfs_list:
    integrated_df = pd.concat(dfs_list, ignore_index=True)
    integrated_df = integrated_df.sort_values(by='측정일시').reset_index(drop=True)
    integrated_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("\n" + "="*60)
    print("🎉 대기질 데이터 통합 대성공!")
    print(f"💾 저장된 파일 경로: {os.path.abspath(output_path)}")
    print("="*60)
else:
    print("🚨 파일 합치기에 실패했습니다. 코랩 왼쪽 파일 목록에 '안산도시1월.xls' 등 파일이 있는지 꼭 확인해 주세요!")

🔍 폴더 내에서 12개의 대기질 파일을 발견했습니다!
   ㄴ [성공] 송내대로 (1).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (10).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (11).xls -> 672행 정제 완료
   ㄴ [성공] 송내대로 (12).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (2).xls -> 720행 정제 완료
   ㄴ [성공] 송내대로 (3).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (4).xls -> 720행 정제 완료
   ㄴ [성공] 송내대로 (5).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (6).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (7).xls -> 720행 정제 완료
   ㄴ [성공] 송내대로 (8).xls -> 744행 정제 완료
   ㄴ [성공] 송내대로 (9).xls -> 720행 정제 완료

🎉 대기질 데이터 통합 대성공!
💾 저장된 파일 경로: /content/송내대로(도시)_2025_1년통합본.csv


# 대기질 데이터 도심 데이터 가중치 재설정 및 주말및 공휴일 배제.

## 성남시

In [ ]:
import pandas as pd
import numpy as np

# 1. 파일 로드 (인코딩 자동 처리)
def load_csv_safe(path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, engine='c')
        except:
            pass
    return pd.read_csv(path, encoding='utf-8-sig', engine='python')

# -------------------------------------------------------------
# 2. [핵심 로직] 단속 데이터에서 '순수 평일' 날짜만 쏙 뽑아내기
# -------------------------------------------------------------
df_enf = load_csv_safe("enforcement_with_grid_id.csv")

# enforced_at (예: 2025-01-01 14:00) 에서 날짜(YYYY-MM-DD)만 추출
df_enf['date_only'] = df_enf['enforced_at'].astype(str).str.split(' ').str[0]

# 주말과 공휴일/명절을 철저히 배제한 '평일'의 날짜 리스트 확보
working_days = df_enf[df_enf['day_type'] == '평일']['date_only'].unique()

# -------------------------------------------------------------
# 3. 대기질 데이터 로드 및 '순수 평일' 필터링
# -------------------------------------------------------------
df_air = load_csv_safe("air_quality_transformed (1).csv")

df_air['clean_at'] = df_air['measured_at'].astype(str).str.replace(' 24:', ' 00:')
df_air['dt'] = pd.to_datetime(df_air['clean_at'])
df_air['date_only'] = df_air['dt'].dt.strftime('%Y-%m-%d')

# ★ 대기질 데이터도 단속 데이터 기준 일하는 '평일'만 남기도록 강력 필터링!
df_air_weekday = df_air[df_air['date_only'].isin(working_days)].copy()

# 4. 프론트엔드 선택 옵션용 [월, 요일, 시간대] 컬럼 생성
df_air_weekday['month'] = df_air_weekday['dt'].dt.month
dow_map = {0:'월', 1:'화', 2:'수', 3:'목', 4:'금', 5:'토', 6:'일'} # 사실상 월~금만 남게 됨
df_air_weekday['day_of_week'] = df_air_weekday['dt'].dt.dayofweek.map(dow_map)
df_air_weekday['hour'] = df_air_weekday['dt'].dt.hour

# -------------------------------------------------------------
# 5. 가중치 보정 (도로변 vs 도시대기) - 오직 평일 데이터로만 산출
# -------------------------------------------------------------
road_df = df_air_weekday[df_air_weekday['station_name'] == '대왕판교로']
bg_df = df_air_weekday[df_air_weekday['station_name'] != '대왕판교로']

grp_cols = ['month', 'day_of_week', 'hour']
pollutants = ['no2', 'co', 'pm10', 'pm25']

road_mean = road_df.groupby(grp_cols)[pollutants].mean().reset_index()
bg_mean = bg_df.groupby(grp_cols)[pollutants].mean().reset_index()

# 월/요일/시간대별 가중치 병합
weights = pd.merge(road_mean, bg_mean, on=grp_cols, suffixes=('_road', '_bg'))

for p in pollutants:
    # 예외 처리: 주거지 수치가 0일 경우 에러 방지 (기본 가중치 1.0 부여)
    weights[f'weight_{p}'] = np.where(
        weights[f'{p}_bg'] > 0,
        weights[f'{p}_road'] / weights[f'{p}_bg'],
        1.0
    )

# -------------------------------------------------------------
# 6. 원본 데이터에 가중치 매핑 및 최종 수치 보정
# -------------------------------------------------------------
df_merged = pd.merge(df_air_weekday, weights[grp_cols + [f'weight_{p}' for p in pollutants]], on=grp_cols, how='left')

for p in pollutants:
    # 도로변(대왕판교로)은 실제 수치 유지, 도시대기는 평일 가중치를 곱해 현실화
    df_merged[p] = np.where(
        df_merged['station_name'] == '대왕판교로',
        df_merged[p],
        df_merged[p] * df_merged[f'weight_{p}']
    )

# 7. 최종 파일 정리 및 저장 (불필요한 계산용 컬럼 제거)
cols_to_drop = [col for col in df_merged.columns if col.startswith('weight_')] + ['clean_at', 'dt', 'date_only']
df_final = df_merged.drop(columns=cols_to_drop)

output_file = "air_quality_corrected_weekday_only.csv"
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 완벽한 '출근일(평일)' 전용 데이터 변환 완료! '{output_file}'이 생성되었습니다.")

🎉 완벽한 '출근일(평일)' 전용 데이터 변환 완료! 'air_quality_corrected_weekday_only.csv'이 생성되었습니다.


## 부천시

In [ ]:
import pandas as pd
import numpy as np

# 1. 파일 로드 및 초기 병합
files = [
    "중2동(도시)_2025_1년통합본.csv",
    "송내대로(도시)_2025_1년통합본.csv",
    "부천소사본동(도로)_2025_1년통합본.csv",
    "부천내동(도로)_2025_1년통합본.csv"
]

# 관측소별 대략적 위경도 맵핑 (필요시 실제 정확한 좌표로 교체)
station_meta = {
    '중2동': {'lat': 37.4939079, 'lng': 126.7700835},
    '송내대로': {'lat': 37.5060355, 'lng': 126.7595460},
    '부천소사본동': {'lat': 37.4800814, 'lng': 126.7999303},
    '부천내동': {'lat': 37.5201728, 'lng': 126.7733045}
}

df_list = []
for f in files:
    station_name = f.split('(')[0]
    try:
        temp_df = pd.read_csv(f, encoding='cp949')
    except:
        temp_df = pd.read_csv(f, encoding='utf-8')

    temp_df['station_name'] = station_name
    df_list.append(temp_df)

df_air = pd.concat(df_list, ignore_index=True)

# 2. 날짜/시간 파싱 및 정렬 (보간법을 위해 시간순 정렬 필수)
def parse_datetime(dt_str):
    date_part, hour_part = dt_str.split(':')
    if hour_part == '24':
        return pd.to_datetime(date_part) + pd.Timedelta(days=1)
    else:
        return pd.to_datetime(f"{date_part} {hour_part}:00:00")

df_air['dt'] = df_air['측정일시'].apply(parse_datetime)
# 관측소별, 시간순으로 완벽하게 정렬
df_air = df_air.sort_values(by=['station_name', 'dt']).reset_index(drop=True)

df_air['date_only'] = df_air['dt'].dt.strftime('%Y-%m-%d')
df_air['month'] = df_air['dt'].dt.month
df_air['hour'] = df_air['dt'].dt.hour
dow_map = {0:'월', 1:'화', 2:'수', 3:'목', 4:'금', 5:'토', 6:'일'}
df_air['day_of_week'] = df_air['dt'].dt.dayofweek.map(dow_map)

pollutants = ['NO2', 'CO']
# 데이터 타입을 명확히 숫자로 변환 (결측치는 NaN으로 처리)
for p in pollutants:
    df_air[p] = pd.to_numeric(df_air[p], errors='coerce')

# =========================================================================
# 🌟 [핵심] 3단계 결측치 방어 로직 (시뮬레이션 블랙홀 완벽 차단)
# =========================================================================
print("🛠️ 결측치(NaN) 3단계 방어 로직 처리를 시작합니다...")

# [1단계] 1~2시간 짧은 누락 ➔ 선형 보간 (Linear Interpolation)
# 동일 관측소 내에서 앞뒤 시간대의 데이터를 선으로 이어 자연스럽게 채움 (최대 2시간 연속 결측치까지만)
for p in pollutants:
    df_air[p] = df_air.groupby('station_name')[p].transform(
        lambda x: x.interpolate(method='linear', limit=2, limit_direction='both')
    )

# [2단계] 반나절~하루 누락 ➔ 자가 평균 대치 (Historical Average)
# 1단계로 못 채운 구멍은 '동일 관측소 + 같은 월 + 같은 요일 + 같은 시간대'의 평균으로 대치
self_mean = df_air.groupby(['station_name', 'month', 'day_of_week', 'hour'])[pollutants].transform('mean')
for p in pollutants:
    df_air[p] = df_air[p].fillna(self_mean[p])

# [3단계] 장기 누락 ➔ 부천시 내 타 지역 관측소 동시간대 평균
# 센서 장기 고장 등으로 1, 2단계를 다 거치고도 비어있는 초장기 결측치는 타 지역 평균으로 대치
other_mean = df_air.groupby(['month', 'day_of_week', 'hour'])[pollutants].transform('mean')
for p in pollutants:
    df_air[p] = df_air[p].fillna(other_mean[p])

# (안전망) 1~3단계를 거치고도 혹시 남은 값이 있다면 전체 평균으로 최후 방어
for p in pollutants:
    df_air[p] = df_air[p].fillna(df_air[p].mean())

print("✅ 결측치 보완 100% 완료!")
# =========================================================================

# 3. '순수 평일' 필터링 (주말 및 2025 법정 공휴일 제거)
holidays_2025 = ['2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',
                 '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06',
                 '2025-08-15', '2025-10-03', '2025-10-06', '2025-10-07', '2025-10-08',
                 '2025-10-09', '2025-12-25']

df_air_weekday = df_air[
    (df_air['dt'].dt.dayofweek <= 4) &
    (~df_air['date_only'].isin(holidays_2025))
].copy()

# 4. [지역별 × 월별 × 시간대별] 3차원 가중치 연산
road_df = df_air_weekday[df_air_weekday['station_name'] == '부천내동']
bg_df = df_air_weekday[df_air_weekday['station_name'] != '부천내동']

grp_cols_road = ['month', 'hour']
grp_cols_bg = ['station_name', 'month', 'hour']

road_mean = road_df.groupby(grp_cols_road)[pollutants].mean().reset_index()
bg_mean = bg_df.groupby(grp_cols_bg)[pollutants].mean().reset_index()

weights = pd.merge(bg_mean, road_mean, on=['month', 'hour'], suffixes=('_bg', '_road'))

for p in pollutants:
    weights[f'weight_{p}'] = np.where(
        weights[f'{p}_bg'] > 0,
        weights[f'{p}_road'] / weights[f'{p}_bg'],
        1.0
    )

# 5. 원본 데이터에 가중치 매핑 및 최종 수치 현실화
df_merged = pd.merge(df_air_weekday,
                     weights[['station_name', 'month', 'hour'] + [f'weight_{p}' for p in pollutants]],
                     on=['station_name', 'month', 'hour'],
                     how='left')

for p in pollutants:
    # 도로변(부천내동)은 가중치가 없으므로 1.0으로 처리하여 원본 값 유지
    df_merged[f'weight_{p}'] = df_merged[f'weight_{p}'].fillna(1.0)
    df_merged[f'adj_{p}'] = df_merged[p] * df_merged[f'weight_{p}']

# 6. 최종 프론트엔드/백엔드 DB용 ERD 규격 맞춤
df_merged['air_id'] = range(1, len(df_merged) + 1)
df_merged['grid_id'] = np.nan
df_merged['lat'] = df_merged['station_name'].map(lambda x: station_meta[x]['lat'])
df_merged['lng'] = df_merged['station_name'].map(lambda x: station_meta[x]['lng'])
df_merged['measured_at'] = df_merged['dt'].dt.strftime('%Y-%m-%d %H:%M')

df_merged['no2'] = df_merged['adj_NO2'].round(4)
df_merged['co'] = df_merged['adj_CO'].round(4)

# 이미지의 컬럼 순서 일치
final_columns = ['air_id', 'grid_id', 'station_name', 'lat', 'lng', 'measured_at', 'month', 'day_of_week', 'hour', 'no2', 'co']
df_final = df_merged[final_columns].sort_values(by=['measured_at', 'station_name'])

# 7. 결과 산출
output_file = "bucheon_air_quality_corrected_erd.csv"
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 부천시 데이터 파이프라인 완벽 가동: {output_file}")

🛠️ 결측치(NaN) 3단계 방어 로직 처리를 시작합니다...
✅ 결측치 보완 100% 완료!
🎉 부천시 데이터 파이프라인 완벽 가동: bucheon_air_quality_corrected_erd.csv


## 안양시 (평촌)

에어코리아 **최종확정 측정자료(시간자료)** 기준. 2025년 12개월 x 2측정소 = 17,520행.
`부림동`(안양시청 민원실)과 `호계3동`(호계복합청사 옥상) 둘 다 **도시대기**이고 둘 다 동안구다.

부천 셀과 다른 점이 둘 있다.

**1. 도로변 보정 생략.** 안양에는 도로변 측정소가 없어 `road_mean`을 만들 수 없다. 부천 계수를
이식하려면 부천 원자료가 필요하고, 그마저도 "부천의 도로변/도시 비율이 안양에도 성립한다"는
미검증 가정을 얹게 되므로 도시대기 실측을 그대로 쓴다. 두 측정소 모두 도시대기라 격자 간
상대 순위는 왜곡되지 않는다. 대신 단속이 일어나는 도로변 대비 환경민감도가 다소 과소평가된다.

> 부천식 보정은 우리 집계 방식과 상성이 나쁘기도 하다. `weight = road_mean(m,h) / bg_mean(s,m,h)`
> 를 곱한 뒤 `(측정소, 월, 시간)`으로 평균 내면 결과가 `road_mean(m,h)`로 수렴해 **측정소별
> 차이가 정확히 상쇄된다.** 연간 집계를 하는 평촌에서는 측정소를 2곳 받은 의미가 사라진다.

**2. 연간 통합.** 평촌은 월별이 아니라 1년치 1건이므로 `(측정소 x 시간대)` 연평균 48행만 낸다.
시간별 원자료를 그대로 두면 `risk._air_for_grids`의 KD-Tree가 `(월, 시간)`마다 좌표가 중복된
수십 행 중 임의의 하루 값 하나만 집어간다. `analysis_month=10`은 "10월"이 아니라 연간 통합을
가리키는 자리표시자다.


In [ ]:
import glob

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ── 안양 평촌 대기질 ───────────────────────────────────────────────────────────
# 에어코리아 최종확정 측정자료(시간자료). 2025년 12개월 x 2측정소 = 17,520행.
#
# 부천 셀과 다른 점 2가지
#  1. 도로변 보정 생략 - 안양 측정소 2곳(부림동·호계3동)이 모두 도시대기라 road_mean을
#     만들 수 없다. 부천 계수를 이식하려면 부천 원자료가 필요하고, 그마저도 "부천의
#     도로변/도시 비율이 안양에도 성립한다"는 미검증 가정을 얹게 되므로 실측을 그대로 쓴다.
#  2. 연간 통합 - 평촌은 월별이 아니라 1년치 1건이다. (측정소 x 시간대) 연평균 48행을 내고
#     analysis_month=10 슬롯에 넣는다. 시간별 원자료를 그대로 두면 risk._air_for_grids의
#     KD-Tree가 (월,시간)마다 임의의 하루 값 하나만 집어간다.

REGION_CODE = 'pyeongchon'
ANALYSIS_YEAR, ANALYSIS_MONTH = 2025, 10
AIR_DIR = '/content/drive/MyDrive/anyang_air_quality'
GRID_PATH = '/content/drive/MyDrive/pyeongchon_grids.csv'
OUT_PATH = '/content/drive/MyDrive/pyeongchon_air_quality.csv'

# 측정소 좌표 - 에어코리아 측정소 정보 기준. 둘 다 도시대기, 둘 다 동안구.
# dir는 원자료가 담긴 폴더명으로, 공식 측정소명과 다를 수 있다(호계동 vs 호계3동).
STATIONS = {
    # 안양시 동안구 시민대로 235 안양시청 민원실
    '부림동': {'dir': '부림동', 'lat': 37.394287, 'lng': 126.956868},
    # 안양시 동안구 경수대로 504 호계복합청사 옥상
    '호계3동': {'dir': '호계동', 'lat': 37.367501, 'lng': 126.958641},
}
POLLUTANTS = ['no2', 'co']
DOW = {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
HOLIDAYS = ['2025-01-01', '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',
            '2025-03-03', '2025-05-05', '2025-05-06', '2025-06-03', '2025-06-06',
            '2025-08-15', '2025-10-03', '2025-10-06', '2025-10-07', '2025-10-08',
            '2025-10-09', '2025-12-25',
            '2026-01-01']  # 12-31 24시가 익일로 넘어가는 행 처리

# 1. 로드 - 파일 안에 측정소명이 없으므로 폴더명으로 구분한다.
frames = []
for station, meta in STATIONS.items():
    paths = sorted(glob.glob(f"{AIR_DIR}/{meta['dir']}/*.xls"))
    assert paths, f"{station}: 원자료를 찾지 못했다 ({AIR_DIR}/{meta['dir']})"
    for path in paths:
        raw = pd.read_excel(path, sheet_name=0, skiprows=[1])  # 1행은 단위 표기
        raw = raw.rename(columns={'이산화질소': 'no2', '일산화탄소': 'co'})[['날짜', 'no2', 'co']]
        raw['station_name'] = station
        frames.append(raw)
df = pd.concat(frames, ignore_index=True)
print(f"로드 {len(df):,}행 / 측정소 {df['station_name'].nunique()}곳")
assert df['station_name'].nunique() == len(STATIONS), '측정소 일부가 누락됐다'


# 2. 날짜 파싱 - 형식은 MM-DD-HH이고 자정을 24시로 표기한다.
#    그날 00시 + 24시간이 곧 익일 00시라 별도 분기 없이 정확히 처리된다.
parts = df['날짜'].astype(str).str.split('-', expand=True).astype(int)
df['dt'] = (pd.to_datetime(dict(year=ANALYSIS_YEAR, month=parts[0], day=parts[1]))
            + pd.to_timedelta(parts[2], unit='h'))
df = df.sort_values(['station_name', 'dt']).reset_index(drop=True)  # 보간 전 시간순 정렬 필수
df['date_only'] = df['dt'].dt.strftime('%Y-%m-%d')
df['month'] = df['dt'].dt.month
df['hour'] = df['dt'].dt.hour
df['day_of_week'] = df['dt'].dt.dayofweek.map(DOW)
for p in POLLUTANTS:
    df[p] = pd.to_numeric(df[p], errors='coerce')
print('보완 전 결측:', {p: int(df[p].isna().sum()) for p in POLLUTANTS})

# 3. 결측 3단계 방어 (부천 셀과 동일)
for p in POLLUTANTS:  # [1] 1~2시간 짧은 누락 -> 선형 보간
    df[p] = df.groupby('station_name')[p].transform(
        lambda x: x.interpolate(method='linear', limit=2, limit_direction='both'))
self_mean = df.groupby(['station_name', 'month', 'day_of_week', 'hour'])[POLLUTANTS].transform('mean')
other_mean = df.groupby(['month', 'day_of_week', 'hour'])[POLLUTANTS].transform('mean')
for p in POLLUTANTS:
    df[p] = df[p].fillna(self_mean[p])   # [2] 자가 과거 평균
    df[p] = df[p].fillna(other_mean[p])  # [3] 타 측정소 동시간대 평균
    df[p] = df[p].fillna(df[p].mean())   # 안전망
assert df[POLLUTANTS].notna().all().all(), '결측 보완 실패'
print('결측 보완 완료')

# 4. 순수 평일만 (주말 + 2025 법정공휴일 배제)
weekday = df[(df['dt'].dt.dayofweek <= 4) & (~df['date_only'].isin(HOLIDAYS))].copy()
print(f'평일 {len(weekday):,}행 ({len(weekday) / len(df):.1%})')

# 5. 도로변 보정 생략 - 사유는 상단 주석 참고.

# 6. 연간 통합 - (측정소 x 시간대) 평균. 12개월이 한 값으로 접힌다.
agg = weekday.groupby(['station_name', 'hour'], as_index=False)[POLLUTANTS].mean()
print(f"연간 집계 {len(agg)}행 = 측정소 {agg['station_name'].nunique()}곳 x 24시간")

# 7. 계약 컬럼 - analysis_month=10은 '10월'이 아니라 '연간 통합' 자리표시자다.
agg['region_code'] = REGION_CODE
agg['lat'] = agg['station_name'].map(lambda s: STATIONS[s]['lat'])
agg['lng'] = agg['station_name'].map(lambda s: STATIONS[s]['lng'])
agg['analysis_month'] = ANALYSIS_MONTH
agg = agg.rename(columns={'hour': 'hour_of_day'})
stamp = pd.Timestamp(ANALYSIS_YEAR, ANALYSIS_MONTH, 1) + pd.to_timedelta(agg['hour_of_day'], unit='h')
agg['measured_at'] = stamp.dt.strftime('%Y-%m-%d %H:%M')
agg['day_of_week'] = stamp.dt.dayofweek.map(DOW)
agg[POLLUTANTS] = agg[POLLUTANTS].round(4)

# 격자 매핑 - 비워두면 파이프라인이 채우지만 여기서 확정해 둔다.
grids = pd.read_csv(GRID_PATH, dtype={'grid_code': str})
_, index = cKDTree(grids[['center_lat', 'center_lng']].to_numpy()).query(agg[['lat', 'lng']].to_numpy())
agg['grid_code'] = grids.iloc[index]['grid_code'].to_numpy()

AIR_COLS = ['region_code', 'grid_code', 'station_name', 'lat', 'lng', 'measured_at',
            'analysis_month', 'day_of_week', 'hour_of_day', 'no2', 'co']
air_out = agg[AIR_COLS].sort_values(['station_name', 'hour_of_day']).reset_index(drop=True)

assert set(air_out['grid_code']) <= set(grids['grid_code']), '격자에 없는 grid_code'
assert len(air_out) == len(STATIONS) * 24, '측정소 x 24시간이 아니다'
assert sorted(air_out['hour_of_day'].unique()) == list(range(24)), '누락된 시간대가 있다'

air_out.to_csv(OUT_PATH, index=False, encoding='utf-8')
print(f'\n계약 CSV 생성 완료: {len(air_out)}행')
print(air_out.groupby('station_name')[POLLUTANTS].agg(['min', 'mean', 'max']).round(4).to_string())


# grid_id와 결합

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 파일 로드 (경로는 코랩 환경에 맞게 수정해 주세요)
air_path = '/content/drive/MyDrive/bucheon_air_quality_corrected_erd.csv'
grid_path = '/content/drive/MyDrive/grids_bucheon_final.csv'

def load_csv_safe(path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    return pd.read_csv(path)

df_air = load_csv_safe(air_path)
df_grid = load_csv_safe(grid_path)

print(f"☁️ 대기질 데이터: {len(df_air):,}건, 격자 데이터: {len(df_grid):,}건 로드 완료")

# 2. 그리드 중심 좌표 기준 KD-Tree 구성
grid_coords = df_grid[['center_lat', 'center_lng']].values
tree = cKDTree(grid_coords)

# 3. 대기질 관측소 좌표를 기반으로 가장 가까운 격자 탐색
air_coords = df_air[['lat', 'lng']].values
distances, indices = tree.query(air_coords)

# 4. 찾은 인덱스를 바탕으로 grid_id 업데이트 (정수형 변환)
df_air['grid_id'] = df_grid.iloc[indices]['grid_id'].values.astype(int)

# 5. 최종 변환된 CSV 저장 (백엔드 적재용 UTF-8-SIG)
output_file = '/content/drive/MyDrive/bucheon_air_quality_grid_mapped.csv'
df_air.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n🎉 대기질 데이터 격자 매핑 완료!")
print(f"👉 파일 저장 위치: {output_file}")
print(df_air[['air_id', 'grid_id', 'station_name', 'lat', 'lng', 'measured_at']].head(5))

☁️ 대기질 데이터: 23,428건, 격자 데이터: 55,256건 로드 완료

🎉 대기질 데이터 격자 매핑 완료!
👉 파일 저장 위치: /content/drive/MyDrive/bucheon_air_quality_grid_mapped.csv
   air_id  grid_id station_name        lat         lng       measured_at
0       1    46474         부천내동  37.520173  126.773304  2025-01-02 00:00
1    5858    15863       부천소사본동  37.480081  126.799930  2025-01-02 00:00
2   11715    35576         송내대로  37.506036  126.759546  2025-01-02 00:00
3   17572    26732          중2동  37.493908  126.770083  2025-01-02 00:00
4       2    46474         부천내동  37.520173  126.773304  2025-01-02 01:00
